In [1]:
import os
import glob
import pathlib
from typing import Annotated
from dotenv import load_dotenv; load_dotenv()

from langchain.tools import tool, ToolRuntime
from langchain.agents import AgentState, create_agent
from langchain.messages import ToolMessage
from langgraph.graph.message import add_messages
from langgraph.types import Command
import warnings; warnings.filterwarnings("ignore")

from langchain.mcp import MCPAdapter

CONFIG = {
    "mcpServers":{
        "search_and_download": {"command": "python", "args": ["/home/rohit_vastani/academic-mcp-agent/mcp_server/arxiv_search.py"]},
        "create_read_store": {"command": "python", "args": ["/home/rohit_vastani/academic-mcp-agent/mcp_server/vec.py"]},
        "context_retriever": {"command": "python", "args": ["/home/rohit_vastani/academic-mcp-agent/mcp_server/search_context.py"]}
    }
}

In [2]:
# async with MCPAdapter(CONFIG) as adapter:
#     tools = await adapter.list_tools()

adapter =  MCPAdapter(CONFIG)

# await adapter.start()

tools = await adapter.list_tools()

[10/28/26 17:06:18] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12982987;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12982988;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:06:30] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12982993;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12982994;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:06:42] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12982999;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983000;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

In [ ]:
class PaperList(AgentState):
    downloaded_papers: list

MAIN_DIR = pathlib.Path(os.path.dirname(__vsc_ipynb_file__)).parent
tmp_dir = MAIN_DIR / "tmp"


In [ ]:
@tool
def update_paper_list(runtime: ToolRuntime) -> Command:
    """Update donwloaded paper list"""

    pdf_files = glob.glob(str(tmp_dir) + "/*.pdf")
    pdf_names = [pathlib.Path(pdf).name for pdf in pdf_files]

    return Command(update={
        "downloaded_papers": pdf_names,
        "messages": [ToolMessage("Successfully updated downloaded paper list", tool_call_id=runtime.tool_call_id)]
    })

@tool
def read_paper_list(runtime: ToolRuntime) -> list | str:
    """Read which research papers are donwloaded"""
    
    papers = runtime.state.get("downloaded_papers", [])
    if not papers:
        return "There are no papers donwloaded"

    return papers

In [12]:
from IPython.display import Markdown, display

def mprint(text: str) -> None:
    display(Markdown(text))

In [ ]:
tools = tools + [update_paper_list, read_paper_list]

In [3]:
tools

[StructuredTool(name='search_and_download_search_paper', description='Finds the exact, foundational research paper matching the query title/topic.\nAfter finding download the research paper to the local system.', args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'title': {'type': 'string'}}, 'required': ['title']}, metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'MCPRouter', 'version': '4.1.3'}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x750da06d3a60>, response_format='content_and_artifact', coroutine=<function as_langchain_tool.<locals>.call_tool at 0x750da008a340>),
 StructuredTool(name='create_read_store_load_and_store_in_vector_database', description='Load the research papers from the directory and split using text splitter.\nAfter splitting the text store it into the chroma vector database', args_schema={'type': 'object', 'additionalProperties': False, 'properties': {}}, metadata={'mcp': {'tool': {'_meta'

In [4]:
system_prompt = """You are a helpful AI assistant which help user to understand research papers.
You have handful tools which can donwload and process and get the relevant context from papers.

Things to consider -
1. If the paper is already downloaded dont donwload again and process it just simply say it's already donwloaded.
2. Give answer only from the given context if the No relevant context found then simply say you dont know.
"""

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    # state_schema=PaperList,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt
)

In [6]:
config = {"configurable": {"thread_id": "2"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="download attention is all you need paper")]},
    config=config
)

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


[10/28/26 17:07:48] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983005;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983006;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:07:53] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983011;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983012;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:08:04] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983017;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983018;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


[10/28/26 17:08:30] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983023;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983024;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:08:36] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983029;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983030;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:08:47] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983035;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983036;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


In [8]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6fb9fb8d-a1fb-4ad9-ba98-3d95c0ebb748'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_and_download_search_paper', 'arguments': '{"title": "attention is all you need"}'}, '__gemini_function_call_thought_signatures__': {'call_7844699': 'EnMKcQFpFH0TDf9nM6ZRs++IZaje3kfP/vN+0gOAuIxQUpLIxmNaojbACMPR5vK9UOLj5h8I6ndDPXOtiigB7N+luJS9WexqIgUEqLFkjuGetIiPueif6sip84DRynexQMZZ7KiD+cv5HhZ95mv+G3HdUNY2'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1824d-3581-76f5-aa76-647f23293b7a-0', tool_calls=[{'name': 'search_and_download_search_paper', 'args': {'title': 'attention is all you need'}, 'id': 'call_7844699', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 300, 'output_tokens': 26, 'total_tokens': 326, '

In [9]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Vaswani talk about BERT in attention is all you need paper ?")]},
    config=config
)

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


[10/28/26 17:12:23] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983041;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983042;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:12:29] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983047;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983048;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/28/26 17:12:40] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12983053;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12983054;file:///home/rohit_vastani/academic-mcp-agent/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1371\1371]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


In [10]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6fb9fb8d-a1fb-4ad9-ba98-3d95c0ebb748'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_and_download_search_paper', 'arguments': '{"title": "attention is all you need"}'}, '__gemini_function_call_thought_signatures__': {'call_7844699': 'EnMKcQFpFH0TDf9nM6ZRs++IZaje3kfP/vN+0gOAuIxQUpLIxmNaojbACMPR5vK9UOLj5h8I6ndDPXOtiigB7N+luJS9WexqIgUEqLFkjuGetIiPueif6sip84DRynexQMZZ7KiD+cv5HhZ95mv+G3HdUNY2'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1824d-3581-76f5-aa76-647f23293b7a-0', tool_calls=[{'name': 'search_and_download_search_paper', 'args': {'title': 'attention is all you need'}, 'id': 'call_7844699', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 300, 'output_tokens': 26, 'total_tokens': 326, '

In [13]:
mprint(response["messages"][-1].content[0]["text"])

No, the paper "Attention Is All You Need" does not discuss BERT. BERT was introduced in later research (specifically by Devlin et al. in 2019), and while it is built upon the Transformer architecture presented in the "Attention Is All You Need" paper, the original 2017 paper does not mention it. Any references to BERT in the provided context appear only in the bibliographies of other related papers.

In [14]:
response["messages"][-2].artifact["structured_content"]["passages"]

[{'text': 'self-attention in transformer models. CoRR, abs/2005.00743, 2020.\n[10] Sinong Wang, Belinda Z. Li, Madian Khabsa, Han Fang, and Hao Ma. Linformer: Self-attention with\nlinear complexity. CoRR, abs/2006.04768, 2020.\n[11] Yihe Dong, Jean-Baptiste Cordonnier, and Andreas Loukas. Attention is not all you need: Pure attention\nloses rank doubly exponentially with depth. CoRR, abs/2103.03404, 2021.\n[12] Saurabh Goyal, Anamitra Roy Choudhury, Saurabh Raje, Venkatesan T. Chakaravarthy, Yogish Sabharwal,\nand Ashish Verma. Power-bert: Accelerating BERT inference via progressive word-vector elimination. In\nProceedings of the 37th International Conference on Machine Learning, ICML 2020, 13-18 July 2020,\nVirtual Event, volume 119 of Proceedings of Machine Learning Research, pages 3690–3699. PMLR, 2020.\n[13] Nitish Srivastava, Geoffrey E. Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov. Dropout:',
  'page': 9,
  'source': '2104.04692v3.pdf',
  'score': 0.75},
 {'t